In [ ]:
import pandas as pd
from scipy.stats import ks_2samp
import numpy as np
import re

from gsm_benchmarker.results_analyser.prompt_result import MultiPromptResult
from gsm_benchmarker.results_analyser.utils import pandas_to_latex

from results_notebook_setup import ALPHA, results_loader, original_model_order, OUTPUTS_FOLDER, significant_models_path


In [ ]:
full_results = results_loader.full_results

In [ ]:
_ = results_loader.gsm.mres.plot_number_counts(save_prefix=OUTPUTS_FOLDER)

### Evaluating number distribution shift in GSM-Variants w.r.t. GSM-Base

In [ ]:
number_counts = results_loader.gsm.mres.get_number_counts()[0]
numeric_index = pd.to_numeric(number_counts.index)

gsm8k_samples = np.repeat(numeric_index, number_counts['GSM-Base'].values)
main_samples = np.repeat(numeric_index, number_counts['GSM-Variants'].values)

ks_stat, p_value = ks_2samp(gsm8k_samples, main_samples)

print(f"K-S Statistic: {ks_stat:.4f}")
print(f"P-value: {p_value:.4e}")

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

In [ ]:
results_loader.gsm.plot_variant_effect()

In [189]:
results_loader.gsm.variant_effect[0]

,estimate,std_err,z_value,p_value,odds_ratio,odds_change,GSM8K_acc,main_acc,acc_diff
model,,,,,,,,,
Mathstral-7B-v0.1,-0.741676,0.343995,-2.156066,3.107850e-02,0.476315,-0.523685,82.0,75.12,-6.88
Meta-Llama-3-8B,-0.696072,0.301905,-2.305596,2.113319e-02,0.498540,-0.501460,55.0,47.44,-7.56
Meta-Llama-3-8B-Instruct,-0.722251,0.327126,-2.207871,2.725327e-02,0.485658,-0.514342,77.0,69.92,-7.08
Mistral-7B-Instruct-v0.1,-0.687820,0.272920,-2.520222,1.172810e-02,0.502671,-0.497329,36.0,27.46,-8.54
Mistral-7B-Instruct-v0.3,-0.500421,0.272969,-1.833250,6.676540e-02,0.606276,-0.393724,52.0,45.32,-6.68
Mistral-7B-v0.1,-0.406632,0.281551,-1.444256,1.486669e-01,0.665889,-0.334111,38.0,33.00,-5.00
Mistral-7B-v0.3,0.206930,0.287684,0.719297,4.719581e-01,1.229897,0.229897,33.0,35.50,2.50
Phi-3-medium-128k-instruct,-0.565484,0.481991,-1.173227,2.407048e-01,0.568085,-0.431915,91.0,88.22,-2.78
Phi-3-mini-128k-instruct,-0.154748,0.351330,-0.440462,6.596022e-01,0.856631,-0.143369,80.0,78.70,-1.30


In [ ]:
results_loader.gsm.variant_effect[1]

In [ ]:
results_loader.gsm.variant_effect_to_latex(model_order=original_model_order)

In [ ]:
significant_models = results_loader.gsm.get_significant_models(alpha=ALPHA, drop_only=True)
significant_models

In [ ]:
len(significant_models)

In [ ]:
for key, res in full_results.items():
    if key != 'GSM':
        res.models = significant_models

with open(significant_models_path, 'w') as f:
    f.writelines([f"{m}\n" for m in significant_models])


## Question 2
Do alternative prompt formats remove the variant dependency?

In [ ]:
results_loader.nonformal.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
results_loader.nonformal.variant_effect_to_latex(model_order=significant_models)

In [ ]:
results_loader.formal.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
results_loader.formal.variant_effect_to_latex(model_order=significant_models)

In [ ]:
results_loader.short_code.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
results_loader.short_code.variant_effect_to_latex(model_order=significant_models)

In [ ]:
results_loader.long_code.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
results_loader.long_code.variant_effect_to_latex(model_order=significant_models)

### Summary of all prompts and models


In [ ]:
all_prompts_result = MultiPromptResult(full_results, save_prefix=OUTPUTS_FOLDER)
all_prompts_result.summary

In [ ]:
# models to be used for plots later - the ones that show significant variant effect on at least one other prompt
s = all_prompts_result.summary
any_sig = s.xs('delta_symb_significant', level=1).iloc[1:].any(axis=0)
any_sig[any_sig].index.tolist()

In [ ]:
# custom ordering for the models identified above
demo_models = [
    'gemma-2b',
    'gemma-2-2b',
    'gemma-7b-it',
    'phi-2',
    'Meta-Llama-3-8B',
    'Meta-Llama-3-8B-Instruct',
]


In [ ]:
fig = all_prompts_result.plot_prompt_comparison(models=demo_models, add_bar_labels=True, x_labels_rotation=5)

In [ ]:
fig = all_prompts_result.plot_prompt_acc_evolution(models=demo_models, n_cols=2, sharex=False, sharey=False, equal_aspect=False, figsize=(10, 10), bottom_margin=.06)


### Number effect tables

In [ ]:
# number effect - GLMM results
all_prompts_result.number_effect_to_latex("number_effect", models=significant_models)

In [ ]:
# number-effect-corrected variant effect - GLMM results
all_prompts_result.number_effect_to_latex("delta_symb_ne", models=significant_models)

### Diagnostics for non-convergent fits

In [ ]:
def make_fit_diagnostics_summary(name):

    all_diagnostics = []
    for prompt_format, res in full_results.items():
        df = getattr(res, name)[1].reset_index()
        df['Prompt'] = prompt_format
        all_diagnostics.append(df)

    combined_diagnostics = pd.concat(all_diagnostics, ignore_index=True)
    total_fits = len(combined_diagnostics)
    clean_fits = combined_diagnostics[
        (~combined_diagnostics['fit_failed']) &
        (~combined_diagnostics['is_singular']) &
        (combined_diagnostics['convergence_messages'] == '')
    ]

    cond = combined_diagnostics['is_singular'] | (combined_diagnostics['convergence_messages'] != '')
    flagged_fits = combined_diagnostics[cond].copy()
    flagged_fits["Diagnostics"] = flagged_fits["convergence_messages"].apply(lambda x: re.search(r"max\|grad\| = \d*\.?\d+ \(.+\)", x).group())
    flagged_fits.set_index("model", inplace=True)

    print(f"{len(clean_fits)} / {total_fits} fits converged cleanly")

    print("\n")
    print(pandas_to_latex(
        flagged_fits[['Prompt', 'Diagnostics']],
        position="H",
        caption=f"Non-convergent fits of {name.replace('_', ' ')} estimation",
    ))
    print()

    if flagged_fits.size:
        print("Convergence messages:")
        for i in range(flagged_fits.shape[0]):
            print(f"#{flagged_fits.index[i]}", flagged_fits.iloc[i].convergence_messages)
            print()

    print("Clean fits summary")
    print(clean_fits['ranef_variance'].agg(['min', 'max', 'mean']))


In [ ]:
make_fit_diagnostics_summary('variant_effect')

In [ ]:
make_fit_diagnostics_summary('number_effect')


In [ ]:
df = results_loader.gsm.mres.get_variant_effect_glmm_data(variant='main', metric='correct')

# per-item, per-model base-vs-variant accuracy
item_compare = (
    df.groupby(['model', 'id', 'is_variant'])['is_correct']
    .mean()
    .unstack('is_variant')
    .rename(columns={0: 'acc_variant_0', 1: 'acc_variant_1'})
)
item_compare['diff'] = item_compare['acc_variant_1'] - item_compare['acc_variant_0']

# collapse to one row per model
item_compare_summary = (
    item_compare['diff']
    .groupby('model')
    .agg(
        mean_diff='mean',
        prop_large_split=lambda s: (s.abs() > 0.8).mean(),
        n_items='size',
    )
)

item_compare_summary

In [ ]:
base_variant_accuracies = df.groupby(['model', 'is_variant'])['is_correct'].mean().unstack('is_variant')

def perc_fmt(precision):
    def wrapper(x):
        px = 100 * x
        if precision == 0:
            return str(int(round(px)))
        return f"{px:.{precision}f}"
    return wrapper

separation_summary_df = pd.DataFrame({
    'GSM-Base acc.': base_variant_accuracies[0].apply(perc_fmt(1)),
    'GSM-Variants acc.': base_variant_accuracies[1].apply(perc_fmt(2)),
    'Mean acc. diff.': item_compare_summary.mean_diff.apply(perc_fmt(2)),
    'Large split': item_compare_summary.prop_large_split.apply(perc_fmt(0)),
    'Constancy': grp.groupby('model').constant.mean().apply(perc_fmt(0)),
}).sort_values('Constancy', ascending=False)

separation_summary_df

In [ ]:

print(pandas_to_latex(
    separation_summary_df,
    position="H",
    caption="Summary of per-model marginal separation and cluster-level degeneracy check. All values given in \%. ``Constancy'' is the proportion of items with degenerate accuracy (all correct or all incorrect for a given template id, pooled across Variants/Base datasets). ``Large split'' is the proportion of items with a difference in accuracy between base and variant greater than 80 percentage points.",
))